In [ ]:
# Feature Engineering (VectorAssembler + Scaler)
from pyspark.ml.feature import VectorAssembler, StandardScaler

exclude_cols = ['ClassLabel', 'Label', 'target_binary', 'target_multiclass',
                'Fwd PSH Flags', 'SYN Flag Count', 'URG Flag Count']

numeric_features = [
    c for c, t in df.dtypes
    if t in ['double', 'float', 'int', 'bigint', 'long']
    and c not in exclude_cols
]

assembler = VectorAssembler(inputCols=numeric_features, outputCol='features_raw', handleInvalid='skip')
df_vec = assembler.transform(df)

scaler = StandardScaler(inputCol='features_raw', outputCol='features', withMean=True, withStd=True)
df_scaled = scaler.fit(df_vec).transform(df_vec).drop('features_raw')

# Sample 30% stratified
df_sample = df_scaled.sampleBy('target_binary', fractions={0: 0.3, 1: 0.3}, seed=42)
train, test = df_sample.randomSplit([0.8, 0.2], seed=42)
train.cache()
test.cache()

print(f'Train: {train.count():,} | Test: {test.count():,}')
print(f'Features: {len(numeric_features)}')

26/02/26 06:36:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 11:===================================================>     (9 + 1) / 10]

Train: 2,201,343 | Test: 550,156
Features: 54


In [ ]:
#Cross Validation with ParamGrid (Random Forest)
from pyspark.ml import Pipeline
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

rf = RandomForestClassifier(featuresCol='features', labelCol='target_binary', seed=42)

# Param grid
param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [20, 50]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

evaluator = MulticlassClassificationEvaluator(
    labelCol='target_binary', predictionCol='prediction', metricName='f1'
)

# 3-Fold Cross Validator
cv = CrossValidator(
    estimator=rf,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

print('Running 3-Fold Cross Validation (4 param combos)... takes ~5-10 mins')
cv_model = cv.fit(train)

print('\nCV Results (avg F1 per param combo):')
for params, score in zip(param_grid, cv_model.avgMetrics):
    print(f'  numTrees={params[rf.numTrees]}, maxDepth={params[rf.maxDepth]} => F1: {score:.4f}')

best_rf = cv_model.bestModel
print(f'\nBest numTrees : {best_rf.getNumTrees}')
print(f'Best maxDepth : {best_rf.getOrDefault(best_rf.maxDepth)}')

Running 3-Fold Cross Validation (4 param combos)... takes ~5-10 mins


26/02/26 06:40:06 WARN DAGScheduler: Broadcasting large task binary with size 1225.1 KiB
26/02/26 06:40:07 WARN DAGScheduler: Broadcasting large task binary with size 1744.1 KiB
26/02/26 06:40:10 WARN DAGScheduler: Broadcasting large task binary with size 1158.9 KiB
26/02/26 06:40:40 WARN DAGScheduler: Broadcasting large task binary with size 1085.6 KiB
26/02/26 06:40:44 WARN DAGScheduler: Broadcasting large task binary with size 1693.5 KiB
26/02/26 06:40:48 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/02/26 06:40:52 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB
26/02/26 06:40:59 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/02/26 06:41:23 WARN DAGScheduler: Broadcasting large task binary with size 1269.4 KiB
26/02/26 06:41:25 WARN DAGScheduler: Broadcasting large task binary with size 1810.0 KiB
26/02/26 06:41:27 WARN DAGScheduler: Broadcasting large task binary with size 1178.0 KiB
26/02/26 06:41:58 WARN DAGSche


CV Results (avg F1 per param combo):
  numTrees=20, maxDepth=5 => F1: 0.9586
  numTrees=20, maxDepth=10 => F1: 0.9812
  numTrees=50, maxDepth=5 => F1: 0.9613
  numTrees=50, maxDepth=10 => F1: 0.9810

Best numTrees : 20
Best maxDepth : 10


In [ ]:
# Hyperparameter Tuning (GBT with TrainValidationSplit)
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.tuning import TrainValidationSplit
from pyspark.ml.evaluation import BinaryClassificationEvaluator

gbt = GBTClassifier(featuresCol='features', labelCol='target_binary', seed=42)

gbt_grid = ParamGridBuilder() \
    .addGrid(gbt.maxIter, [10, 20]) \
    .addGrid(gbt.maxDepth, [5, 8]) \
    .addGrid(gbt.stepSize, [0.1, 0.05]) \
    .build()

auc_eval = BinaryClassificationEvaluator(
    labelCol='target_binary', rawPredictionCol='rawPrediction', metricName='areaUnderROC'
)

tvs = TrainValidationSplit(
    estimator=gbt,
    estimatorParamMaps=gbt_grid,
    evaluator=auc_eval,
    trainRatio=0.8,
    seed=42
)

print('Running GBT Hyperparameter Tuning (8 combos)... takes ~10-15 mins')
tvs_model = tvs.fit(train)

best_gbt = tvs_model.bestModel
pred_gbt = best_gbt.transform(test)
gbt_auc  = auc_eval.evaluate(pred_gbt)
gbt_f1   = evaluator.evaluate(pred_gbt)

print(f'\nBest GBT => AUC: {gbt_auc:.4f} | F1: {gbt_f1:.4f}')
print(f'maxIter : {best_gbt.getMaxIter()}')
print(f'maxDepth: {best_gbt.getMaxDepth()}')
print(f'stepSize: {best_gbt.getStepSize()}')

Running GBT Hyperparameter Tuning (8 combos)... takes ~10-15 mins



Best GBT => AUC: 0.9896 | F1: 0.9858
maxIter : 20
maxDepth: 8
stepSize: 0.1


In [ ]:
# Model Serialization (Save & Load)
import os

MODEL_PATH_RF  = '/mnt/bigdata/models/best_rf_model'
MODEL_PATH_GBT = '/mnt/bigdata/models/best_gbt_model'

os.makedirs('/mnt/bigdata/models', exist_ok=True)

# Save models
best_rf.save(MODEL_PATH_RF)
best_gbt.save(MODEL_PATH_GBT)
print('Models saved!')

# Reload and verify
from pyspark.ml.classification import RandomForestClassificationModel, GBTClassificationModel

loaded_rf  = RandomForestClassificationModel.load(MODEL_PATH_RF)
loaded_gbt = GBTClassificationModel.load(MODEL_PATH_GBT)

pred_rf_loaded  = loaded_rf.transform(test)
pred_gbt_loaded = loaded_gbt.transform(test)

rf_f1  = evaluator.evaluate(pred_rf_loaded)
gbt_f1 = evaluator.evaluate(pred_gbt_loaded)

print(f'Loaded RF  F1 : {rf_f1:.4f}')
print(f'Loaded GBT F1 : {gbt_f1:.4f}')
print('Serialization verified!')

Models saved!


26/02/26 06:50:33 WARN DAGScheduler: Broadcasting large task binary with size 1180.5 KiB


Loaded RF  F1 : 0.9812
Loaded GBT F1 : 0.9858
Serialization verified!


In [ ]:
# Statistical Significance Testing (Paired t-test)
import numpy as np
from scipy import stats

# Get per-sample correctness for RF and GBT
from pyspark.sql.functions import col

pred_rf_vals  = pred_rf_loaded.select(
    ((col('prediction') == col('target_binary')).cast('int')).alias('correct')
).rdd.map(lambda r: r[0]).collect()

pred_gbt_vals = pred_gbt_loaded.select(
    ((col('prediction') == col('target_binary')).cast('int')).alias('correct')
).rdd.map(lambda r: r[0]).collect()

rf_scores  = np.array(pred_rf_vals)
gbt_scores = np.array(pred_gbt_vals)

# Paired t-test
t_stat, p_value = stats.ttest_rel(rf_scores, gbt_scores)


print('  Statistical Significance Test (Paired t-test)')
print(f'  RF  Mean Accuracy : {rf_scores.mean():.4f}')
print(f'  GBT Mean Accuracy : {gbt_scores.mean():.4f}')
print(f'  t-statistic       : {t_stat:.4f}')
print(f'  p-value           : {p_value:.6f}')
print()
if p_value < 0.05:
    print('  RESULT: Significant difference (p < 0.05) — GBT is statistically better!')
else:
    print('  RESULT: No significant difference (p >= 0.05) — models are statistically equal.')

26/02/26 06:51:01 WARN DAGScheduler: Broadcasting large task binary with size 1165.9 KiB
                                                                                

  Statistical Significance Test (Paired t-test)
  RF  Mean Accuracy : 0.9814
  GBT Mean Accuracy : 0.9860
  t-statistic       : -40.0731
  p-value           : 0.000000

  RESULT: Significant difference (p < 0.05) — GBT is statistically better!
